# <center> Lecture15 : 课程回顾和复习 </center>  
 
## <center> Instructor: Dr. Hu Chuan-Peng </center> 

## Outlines  

| 序号  |                    课程内容                     |  
| :--: | :--------------------------------------------: |  
|  1   |                    课程介绍                     |  
|  2   |                  Bayes' Rule                   |  
|  3   |        The Beta-Binomial Bayesian Model        |  
|  4   | Balance and Sequentiality in Bayesian Analyses |  
|  5   |          Approximating the Posterior           |  
|  6   |              MCMC under the Hood               |  
|  7   |        Posterior Inference & Prediction        |  
|  8   |           A Simple Normal Regression           |  
|  9   |                  Bayes factors                 |  
|  10  |              Multiple regression               |  
|  11  |         Evaluating Regression Models           |  
|  12  |            GLM: Logistic Regression            |  
|  13  |             Hierarchical Models 1              |  
|  14  |             Hierarchical Models 2              |

## 对本课程最简略的概括  

- 一个概念：贝叶斯视角下的参数  
- 一个公式：贝叶斯公式  
- 一个算法：马尔科夫链蒙特卡洛（MCMC）  
- 一个软件：PyMC  
- 一个workflow：贝叶斯分析的工作流程  


## 一个概念  
**模型参数是随机的，是概率分布**  





## 一个公式  

$$  
P(A|B) = \frac{P(A) * P (B | A)}{P(B)}  
$$  

- $P(A|B)$: 后验概率  
- $P(A)$: 先验概率  
- $P(B|A)$: 似然函数  
- $P(B)$: Marginal likelihood  

<div style="padding-bottom: 50px;"></div>

## 一个算法  
马尔科夫链蒙特卡洛（MCMC）

## 一个软件包  
**PyMC**  

通过PyMC学习使用概率编程语言(probability programming language)，实现贝叶斯推断。  

![Image Name](https://cdn.kesci.com/upload/sl1bdlkzgo.png?imageView2/0/w/640/h/640)  


## 一个数据分析流程  

**Bayesian Workflow**  

![Image Name](https://cdn.kesci.com/upload/sozk5mh1vf.png?imageView2/0/w/960/h/960)  

其他相关指南参考：  

> Kruschke, J.K. Bayesian Analysis Reporting Guidelines. Nat Hum Behav 5, 1282–1291 (2021). https://doi.org/10.1038/s41562-021-01177-7  


## 一个完整的例子  

### 研究问题  

“**随机点运动范式中，反应时间如何受到随机点运动方向一致性比例的影响，如果会的话，其影响程度是怎么样的**？”

### 数据  

Evans et al.（2020, Exp. 1） 的数据，包括57名被试的数据，单因素被试内实验设计，自变量为2个水平。  

<center>  
    <table>  
            <tr>  
                <td><img src="https://cdn.kesci.com/upload/sjwnyi477j.gif?imageView2/0/w/400/h/400" alt=""></td>  
                <td><img src="https://cdn.kesci.com/upload/sjwnyt1yq4.gif?imageView2/0/w/400/h/400" alt=""></td>  
            </tr>  
            <tr>  
                <td>一致性5%</td>  
                <td>一致性10%</td>  
            </tr>  
    </table>  
</center>  

> Evans, N. J., Hawkins, G. E., & Brown, S. D. (2020). The role of passing time in decision-making. Journal of Experimental Psychology: Learning, Memory, and Cognition, 46(2), 316–326. https://doi.org/10.1037/xlm0000725  


In [1]:
# 导入 pymc 模型包，和 arviz 等分析工具 
import pymc as pm
import arviz as az
import seaborn as sns
import scipy.stats as st
import numpy as np
import matplotlib.pyplot as plt
import xarray as xr
import pandas as pd
import ipywidgets
import bambi as bmb

# 忽略不必要的警告
import warnings
warnings.filterwarnings("ignore")


In [2]:
# 使用 pandas 导入示例数据
try:
  df_raw  = pd.read_csv("/home/mw/input/bayes3797/evans2020JExpPsycholLearn_exp1_full_data.csv") 
except:
  df_raw  = pd.read_csv('data/evans2020JExpPsycholLearn_exp1_full_data.csv')

# 筛选出特定被试并创建索引
df = df_raw.query("percentCoherence in [5, 10]").copy()
df["Coherence"] = np.where(df["percentCoherence"] == 5, 0, 1)

# 随机选择30个被试
selected_subjects = df['subject'].drop_duplicates().sample(n=30, random_state=42)
df = df[df['subject'].isin(selected_subjects)]

# 为每个被试建立索引 'subj_id' 和 'obs_id'
df['subj_id'] = df['subject']
df['obs_id'] = df.groupby('subject').cumcount() + 1

# 对反应时间取对数
df["log_RTs"] = np.log(df["RT"])

# 为每一行生成全局唯一编号 'global_id'
df['global_id'] = range(len(df))

df.head()

,subject,blkNum,trlNum,coherentDots,numberofDots,percentCoherence,winningDirection,response,correct,eventCount,averageFrameRate,RT,Coherence,subj_id,obs_id,log_RTs,global_id
0,31727,2,1,4,40,10,right,right,1,51,14.452,3529,1,31727,1,8.168770,0
2,31727,2,3,2,40,5,left,right,0,15,15.322,979,0,31727,2,6.886532,1
6,31727,2,7,4,40,10,right,right,1,48,14.307,3355,1,31727,3,8.118207,2
7,31727,2,8,4,40,10,right,right,1,31,14.472,2142,1,31727,4,7.669495,3
9,31727,2,10,4,40,10,right,right,1,16,15.166,1055,1,31727,5,6.961296,4


### 模型设定  

建立三个不同的模型来探讨反应时间与一致性比例之间的关系。  

#### 模型1：完全池化模型  

**目的：不考虑随机点运动方向一致性的比例对反应时间的影响**  


$$  
\begin{array}{lcrl}  
Y_i | \beta_0, \beta_1, \sigma & \stackrel{ind}{\sim} N\left(\mu_i, \sigma^2\right) \;\; \text{ with } \;\; \mu_i = \beta_0 + \beta_1X_i \\  
\end{array}  
$$  


#### 模型2：部分池化模型（变化截距）  

**目的：随机点运动方向的一致性与反应时间之间的关系在被试内有什么不同**  

$$  
\begin{array}{rll}  
&\mu_{\beta_0}, \sigma_{\beta_0} & \text{Layer 3: 总体水平} \\  
\beta_{0j} | \mu_{\beta_0}, \sigma_{\beta_0}  & \stackrel{\text{ind}}{\sim} N(\mu_{\beta_0}, \sigma_{\beta_0}^2)  & \text{Layer 2: 组水平} \\  
Y_{ij} | \beta_{0j}, \beta_{1}, \sigma_y & \sim N(\mu_{ij}, \sigma_y^2) \;\; \text{ with } \;\;  \mu_{ij} = \beta_{0j}  + \beta_{1} X_{ij} & \text{Layer 1: 试次水平/数据点} \\  
\end{array}  
$$  

#### 模型3：部分池化模型（变化斜率和截距）  

**目的：考虑截距和斜率共同变化的情况，并全局参数进行定义，即**                         

$$  
\begin{array}{rll}  

&\mu_{\beta_0}, \mu_{\beta_1}, \sigma_{\beta_0}, \sigma_{\beta_1} & \text{Layer 3: 总体水平} \\  
\beta_{0j} | \mu_{\beta_0}, \sigma_{\beta_0}  & \stackrel{ind}{\sim} N(\mu_{\beta_0}, \sigma_{\beta_0}^2) & \text{Layer 2: 组水平（截距在被试间的变化）} \\  
\beta_{1j} | \mu_{\beta_1}, \sigma_{\beta_1}  & \stackrel{ind}{\sim} N(\mu_{\beta_1}, \sigma_{\beta_1}^2) & \text{Layer 2: 组水平（斜率在被试间的变化）} \\  
Y_{ij} | \beta_{0j}, \beta_{1j}, \sigma_y & \sim N(\mu_{ij}, \sigma_y^2) \;\; \text{ with } \;\; \mu_{ij} = \beta_{0j} + \beta_{1j} X_{ij} & \text{Layer 1: 试次水平/数据点} \\  
\end{array}  
$$  




In [3]:
## 模型1：完全池化模型
complete_pooled_model = bmb.Model("log_RTs ~ 1 + Coherence", df)

In [4]:
## 模型2：随机截距模型
var_inter_model = bmb.Model("log_RTs ~ 1 + Coherence + (1|subj_id)", df)

In [5]:
# 模型3：随机截距和斜率模型   
var_both_model = bmb.Model("log_RTs ~ Coherence + (Coherence|subj_id)",df)

### 拟合数据及MCMC评估  

接下来会对3个模型进行数据拟合、MCMC评估及后验计算。

**模型1**：完全池化模型

In [6]:
complete_pooled_trace = complete_pooled_model.fit(idata_kwargs={"log_likelihood": True})

Sampling 4 chains, 0 divergences ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100% 0:00:00 / 0:00:04

Sampling 4 chains for 1_000 tune and 1_000 draw iterations (4_000 + 4_000 draws total) took 5 seconds.


In [7]:
complete_pooled_para = az.summary(complete_pooled_trace)
complete_pooled_para

,mean,sd,hdi_3%,hdi_97%,mcse_mean,mcse_sd,ess_bulk,ess_tail,r_hat
Coherence,-0.106,0.009,-0.122,-0.089,0.0,0.0,5318.0,3042.0,1.0
Intercept,6.919,0.006,6.908,6.931,0.0,0.0,5779.0,3208.0,1.0
log_RTs_sigma,0.563,0.003,0.557,0.568,0.0,0.0,5989.0,2560.0,1.0


In [8]:
az.plot_forest(
    complete_pooled_trace,
    var_names=["Coherence"],
    filter_vars="like",
    combined = True)

array([<AxesSubplot: title={'center': '94.0% HDI'}>], dtype=object)

<Figure size 600x410 with 1 Axes>

模型2：部分池化模型（变化截距）

In [9]:
var_inter_trace = var_inter_model.fit(idata_kwargs={"log_likelihood": True})

Sampling 4 chains, 0 divergences ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100% 0:00:00 / 0:00:39

Sampling 4 chains for 1_000 tune and 1_000 draw iterations (4_000 + 4_000 draws total) took 40 seconds.
The rhat statistic is larger than 1.01 for some parameters. This indicates problems during sampling. See https://arxiv.org/abs/1903.08008 for details
The effective sample size per chain is smaller than 100 for some parameters.  A higher number is needed for reliable rhat and ess computation. See https://arxiv.org/abs/1903.08008 for details


In [10]:
var_inter_para = az.summary(var_inter_trace)
var_inter_para 

,mean,sd,hdi_3%,hdi_97%,mcse_mean,mcse_sd,ess_bulk,ess_tail,r_hat
1|subj_id[31727],-0.103,0.064,-0.215,0.029,0.005,0.003,199.0,321.0,1.01
1|subj_id[71329],-0.238,0.064,-0.354,-0.111,0.005,0.003,195.0,290.0,1.01
1|subj_id[71737],0.292,0.064,0.179,0.426,0.005,0.003,204.0,299.0,1.01
1|subj_id[75445],0.488,0.066,0.369,0.616,0.005,0.003,215.0,335.0,1.01
1|subj_id[77704],-0.212,0.064,-0.324,-0.084,0.005,0.003,201.0,316.0,1.01
1|subj_id[79861],-0.380,0.064,-0.498,-0.259,0.005,0.003,198.0,297.0,1.01
1|subj_id[80035],-0.166,0.065,-0.289,-0.042,0.005,0.003,198.0,309.0,1.01
1|subj_id[80362],-0.131,0.065,-0.245,0.000,0.005,0.003,205.0,336.0,1.01
1|subj_id[80446],-0.059,0.064,-0.171,0.070,0.005,0.003,198.0,319.0,1.01
1|subj_id[80611],0.038,0.065,-0.080,0.169,0.005,0.003,201.0,313.0,1.01


In [11]:
az.plot_forest(var_inter_trace,
           var_names=["~Intercept", "~sigma"],
           filter_vars="like",
           combined = True)

array([<AxesSubplot: title={'center': '94.0% HDI'}>], dtype=object)

<Figure size 600x810 with 1 Axes>

**模型3**：部分池化模型（变化截距和斜率）

In [12]:
var_both_trace = var_both_model.fit(idata_kwargs={"log_likelihood": True})

Sampling 4 chains, 0 divergences ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100% 0:00:00 / 0:01:58

Sampling 4 chains for 1_000 tune and 1_000 draw iterations (4_000 + 4_000 draws total) took 118 seconds.


In [13]:
var_both_para = az.summary(var_both_trace)
var_both_para

,mean,sd,hdi_3%,hdi_97%,mcse_mean,mcse_sd,ess_bulk,ess_tail,r_hat
1|subj_id[31727],-0.111,0.069,-0.239,0.022,0.003,0.002,497.0,725.0,1.01
1|subj_id[71329],-0.265,0.069,-0.395,-0.138,0.003,0.002,475.0,839.0,1.01
1|subj_id[71737],0.340,0.069,0.215,0.476,0.003,0.002,515.0,805.0,1.01
1|subj_id[75445],0.446,0.071,0.314,0.579,0.003,0.002,559.0,915.0,1.01
1|subj_id[77704],-0.263,0.069,-0.392,-0.132,0.003,0.002,500.0,823.0,1.01
...,...,...,...,...,...,...,...,...,...
Coherence|subj_id[84322],0.046,0.033,-0.019,0.105,0.001,0.000,3262.0,2778.0,1.00
Coherence|subj_id[84370],-0.128,0.052,-0.221,-0.026,0.001,0.001,4130.0,2625.0,1.00
Coherence|subj_id_sigma,0.090,0.016,0.060,0.120,0.000,0.000,1528.0,2414.0,1.00
Intercept,6.971,0.063,6.852,7.090,0.003,0.002,431.0,626.0,1.01


In [14]:
# 设置绘图坐标
figs, (ax1, ax2) = plt.subplots(1,2, figsize = (20,5))
# 绘制变化的截距
az.plot_forest(var_both_trace,
           var_names=["Coherence\\|subj_id", "~sigma", "~1|", "~Intercept"],
           filter_vars="like",
           combined = True,
           ax=ax1)
# 绘制变化的斜率
az.plot_forest(var_both_trace,
           var_names=["1|subj_id", "~sigma", "~Coherence", "~Intercept"],
           filter_vars="like",
           combined = True,
           ax=ax2)
plt.show()

<Figure size 2000x500 with 2 Axes>

### 模型比较  

##### 模型评估指标  

在分析模型的预测能力时，有绝对指标和相对指标，绝对指标用于衡量模型预测值与真实值之间的差异，相对指标用于比较不同模型的预测能力，通常用于不同方法或模型之间的性能对比。  

##### 绝对指标：  

* 在之前的课程中介绍过对后验预测结果进行评估的两种方法  

* 一是**MAE**，即后验预测值与真实值之间预测误差的中位数，二是**within_95**，即真实值是否落在95%后验预测区间内  

* 在这里调用之前写过的计算两种指标的方法，评估两种分层模型的后验预测结果  


##### 相对指标  

在实际操作中，我们通过 `ArViz` 的函数`az.loo`计算 $ELPD_{LOO-CV}$。  

PSIS-LOO-CV 有两大优势：  
1. 计算速度快，且结果稳健  
2. 提供了丰富的模型诊断指标  

In [15]:
complete_pooled_model.predict(complete_pooled_trace, kind="pps")
var_inter_model.predict(var_inter_trace, kind="pps")
var_both_model.predict(var_both_trace, kind="pps")

In [16]:
complete_pooled_trace

Inference data with groups:
	> posterior
	> posterior_predictive
	> log_likelihood
	> sample_stats
	> observed_data

In [17]:
# 定义计算 MAE 函数
from statistics import median
def MAE(trace):
    # 计算每个X取值下对应的后验预测模型的均值
    pre_x = trace.posterior_predictive["log_RTs"].stack(sample=("chain", "draw"))
    pre_y_mean = pre_x.mean(axis=1).values

    # 提取观测值Y，提取对应Y值下的后验预测模型的均值
    MAE = pd.DataFrame({
        "ppc_mean": pre_y_mean,
        "original": trace.observed_data.log_RTs.values
    })

    # 计算预测误差
    MAE["pre_error"] = abs(MAE["original"] -\
                            MAE["ppc_mean"])

    # 最后，计算预测误差的中位数
    MAE = median(MAE.pre_error)
    return MAE

In [18]:
# 定义
def counter_outlier(model_trace, hdi_prob=0.95):
    # 将az.summary生成的结果存到hdi_multi这个变量中，该变量为数据框
    hdi = az.summary(model_trace.posterior_predictive, kind="stats", hdi_prob=hdi_prob)
    lower = hdi.iloc[:,2].values
    upper = hdi.iloc[:,3].values

    # 将原数据中的自我控制分数合并，便于后续进行判断
    y_obs = model_trace.observed_data["log_RTs"].values

    # 判断原数据中的压力分数是否在后验预测的95%可信区间内，并计数
    hdi["verify"] = (y_obs <= lower) | (y_obs >= upper)
    hdi["y_obs"] = y_obs
    hdi_num = sum(hdi["verify"])

    return hdi_num

In [19]:
# 将每个模型的PPC储存为列表
ppc_samples_list = [complete_pooled_trace, var_inter_trace, var_both_trace]
model_names = ["完全池化", "变化截距", "变化截距、斜率"]

# 建立一个空列表来存储结果
results_list = []

# 遍历模型并计算MAE和超出95%hdi的值
for model_name, ppc_samples in zip(model_names, ppc_samples_list):
    outliers = counter_outlier(ppc_samples)
    MAEs = MAE(ppc_samples)
    results_list.append({'Model': model_name, 'MAE':MAEs, 'Outliers': outliers})

# 从结果列表创建一个DataFrame
results_df = pd.DataFrame(results_list)

results_df

,Model,MAE,Outliers
0,完全池化,0.350056,916
1,变化截距,0.299881,928
2,变化截距、斜率,0.298411,925


In [20]:
comparison_list = {
    "model1(complete pooling)":complete_pooled_trace,
    "model2(hierarchical intercept)":var_inter_trace,
    "model3(hierarchy both)":var_both_trace,
}
az.compare(comparison_list)

,rank,elpd_loo,p_loo,elpd_diff,weight,se,dse,warning,scale
model3(hierarchy both),0,-11583.592544,60.674135,0.000000,0.930974,117.924576,0.000000,False,log
model2(hierarchical intercept),1,-11622.699552,34.783179,39.107008,0.000000,117.418582,9.810850,False,log
model1(complete pooling),2,-13692.756653,3.528592,2109.164108,0.069026,113.485701,72.273518,False,log


通过 `arviz.compare` 方法来对比多个模型的 elpd。从下面结果可见：  
 
- 模型3的 elpd_loo 最大，表明它对**样本外数据**的预测性能最好。  
- 而模型1的 elpd_loo 最小，表明它的预测性能最差。  

因此，根据模型评估的结果，可以发现模型3（变化截距和变化斜率）的结果在3个模型中是最好的。

### 贝叶斯统计推断  

模型比较发现，模型3（变化截距和变化斜率）的结果在3个模型中是最好的。最后，将使用 HDI + ROPE 和贝叶斯因子（Bayes Factor，BF）来进行统计推断。  

#### HDI + ROPE 的统计推断  

* 反应时差异的后验分布平均值为 -281 ms，然而，这一数值并不足以断定实验条件对反应时间有显著的减少作用。  
* 95% HDI 范围为 [-608 ms, 48 ms]，表明后验分布中95%的概率下的反应时差异位于此区间。由于95% HDI 包含了0，并且分布主要集中在负值方向，但这一趋势并不足以证明存在显著的效应。  
* ROPE 设定了一个 [-30 ms, 30 ms] 的实用等效区间，用以判断反应时差异是否具有实际意义。ROPE 内的概率仅为1.6%，尽管这表明在大多数情况下反应时差异超出了可忽略的范围，但这一差异仍不足以被视为显著。  


#### 贝叶斯因子（Bayes Factor，BF）  

反应时的差异（beta_1）在统计上和实际意义上均不显著。  
* 数据强烈支持 无效假设（beta_1 = 0），即反应时差异可能不存在或非常微弱。  
* 贝叶斯因子 BF_10 = 0.01 提供了明确的证据，表明 beta_1 不显著。  
* 从后验分布来看，数据更新后 beta_1 的可能值仍然集中在 0 附近，进一步支持无效假设。  


In [21]:
# 从贝叶斯模型的后验分布中提取参数
def inv_log(mu, sigma):
    return np.exp(mu + (sigma ** 2) / 2)

Intercept_mu = var_both_trace.posterior.stack(sample=("chain", "draw")).get("Intercept")
Coherence_mu = var_both_trace.posterior.stack(sample=("chain", "draw")).get("Coherence")
Intercept_sigma = var_both_trace.posterior.stack(sample=("chain", "draw")).get("1|subj_id_sigma")
Coherence_sigma = var_both_trace.posterior.stack(sample=("chain", "draw")).get("Coherence|subj_id_sigma")

# 计算两个条件下的反应时间
rt_coh_5 = inv_log(Intercept_mu, Intercept_sigma)
rt_coh_10 = inv_log(Intercept_mu+Coherence_mu, Coherence_sigma)
rt_coh_5 = rt_coh_5[(rt_coh_5 >= 300) & (rt_coh_5 <= 1500)]
rt_coh_10 = rt_coh_10[(rt_coh_10 >= 300) & (rt_coh_10 <= 1500)]

rt_diff = rt_coh_10 - rt_coh_5
rt_diff = rt_diff.values

# 定义 ROPE 区间，根据研究的需要指定实际等效范围
rope_interval = [-30, 30]

# 绘制后验分布，显示 HDI 和 ROPE
az.plot_posterior(
    {"RT Difference":rt_diff},
    hdi_prob=0.95,
    rope=rope_interval,
    figsize=(8, 5),
    textsize=12
)

plt.show()

<Figure size 800x500 with 1 Axes>

In [22]:
# 进行贝叶斯因子计算，需要采样先验分布
var_both_trace.extend(var_both_model.prior_predictive(random_seed=84735) )

# 绘制贝叶斯因子图
az.plot_bf(var_both_trace, var_name="Coherence", ref_val=0)

# 设置 x 轴的范围
plt.xlim(-0.5, 0.5) 

# 去除上框线和右框线
sns.despine()

Sampling: [1|subj_id_offset, 1|subj_id_sigma, Coherence, Coherence|subj_id_offset, Coherence|subj_id_sigma, Intercept, log_RTs, log_RTs_sigma]
The reference value is outside of the posterior. This translate into infinite support for H1, which is most likely an overstatement.


<Figure size 640x480 with 1 Axes>

### 结论  


通过模型建立和模型比较，相比于模型1和模型2，模型3（变化截距和变化斜率）的效果最好。因此，在随机点运动范式中，反应时间显著受到随机点运动方向一致性比例的影响。  

具体而言，模型3考虑了变化截距和变化斜率，这意味着它不仅捕捉到了不同条件下反应时间的平均差异，还考虑了这些差异随条件变化的趋势。相比之下，模型1和模型2未能充分捕捉到这些复杂的变化模式，因而在预测精度和模型拟合度上表现不如模型3。  


### 大作业注意事项  

在完成大作业时，请注意以下几点要求：  

1. **文档**：  
   - 请按照APA7论文格式撰写文档，确保内容完整、格式规范。  
   - 文档应包括研究背景、方法、结果和讨论等部分，详细描述研究过程和发现。  

2. **和鲸Notebook演示（或PPT）**：  
   - 使用和鲸Notebook或PPT进行演示，清晰展示研究的各个步骤和结果。  
   - 演示内容应包括数据处理、模型构建、结果分析和结论等部分。  

3. **代码**：  
   - 提交完整的代码，确保代码可以运行并生成预期结果。  
   - 代码应包括数据导入、预处理、模型构建、拟合和结果分析等部分。  
   - 请在代码中添加必要的注释。  


💡互评环节：  

在2025年1月3号的最后一次课时，各小组将进行大作业汇报。每组可以对其他汇报组进行评分，评分标准包括文档规范性、演示效果和代码运行情况等。  

